# Flow Map Data Prep
Prepares OD flow CSVs for kepler.gl visualization.
Upload the output CSVs to https://kepler.gl to make the flow map.

Outputs:
- `outputs/kepler_pre_covid.csv` — Pre-COVID flows (2018-2019)
- `outputs/kepler_covid.csv`     — COVID flows (2020-2021)

In [ ]:
import pandas as pd
import geopandas as gpd
import pygris
from pathlib import Path

FLOW_FILE = Path('../data/processed/flow_map.csv')

DMV_COUNTIES = [
    '11001',
    '24031', '24033',
    '51013', '51059',
    '51510', '51600', '51610',
]

## 1. Load CBG Centroids

In [35]:
block_groups = [pygris.block_groups(state=s, year=2019, cache=True) for s in ['DC', 'MD', 'VA']]
cbgs  = pd.concat(block_groups, ignore_index=True)
cbgs  = gpd.GeoDataFrame(cbgs, crs=cbgs.crs)

# Filter to DMV
cbgs  = cbgs[cbgs['GEOID'].str[:5].isin(DMV_COUNTIES)][['GEOID', 'geometry']].copy()

# Project to meters for accurate centroid, then back to lat/lon
cbgs_projected = cbgs.to_crs('EPSG:32618')  # UTM Zone 18N — covers DMV
cbgs['lng'] = cbgs_projected.geometry.centroid.to_crs('EPSG:4326').x
cbgs['lat'] = cbgs_projected.geometry.centroid.to_crs('EPSG:4326').y

centroids = cbgs[['GEOID', 'lat', 'lng']].copy()

Using FIPS code '11' for input 'DC'
Using FIPS code '24' for input 'MD'
Using FIPS code '51' for input 'VA'


## 2. Load Flows and Join Coordinates

In [36]:
flows = pd.read_csv(FLOW_FILE, dtype={'home_cbg': str, 'poi_cbg': str})
flows['home_cbg'] = flows['home_cbg'].str.zfill(12)
flows['poi_cbg']  = flows['poi_cbg'].str.zfill(12)

# Filter to comparison periods only
flows = flows[flows['period'].isin(['2018-2019 (Pre-COVID)', '2020-2021 (COVID)'])].copy()

# Join origin coordinates
flows = flows.merge(
    centroids.rename(columns={'GEOID': 'home_cbg', 'lat': 'origin_lat', 'lng': 'origin_lng'}),
    on='home_cbg', how='left'
)

# Join destination coordinates
flows = flows.merge(
    centroids.rename(columns={'GEOID': 'poi_cbg', 'lat': 'dest_lat', 'lng': 'dest_lng'}),
    on='poi_cbg', how='left'
)

# Drop rows missing coordinates
flows = flows.dropna(subset=['origin_lat', 'origin_lng', 'dest_lat', 'dest_lng'])

# Remove self-loops — home and destination are the same CBG
flows = flows[flows['home_cbg'] != flows['poi_cbg']].copy()

print(flows[['home_cbg','poi_cbg','period','origin_lat','origin_lng',
             'dest_lat','dest_lng','avg_visitor_count','avg_median_dwell']].head())

C:\Users\sonji\AppData\Local\Temp\ipykernel_13060\2644514081.py:1: DtypeWarning: Columns (0: period) have mixed types. Specify dtype option on import or set low_memory=False.
  flows = pd.read_csv(FLOW_FILE, dtype={'home_cbg': str, 'poi_cbg': str})


        home_cbg       poi_cbg                 period  origin_lat  origin_lng  \
10  240317002051  240317003041  2018-2019 (Pre-COVID)   39.225276  -77.226539   
15  240338072002  240338072003  2018-2019 (Pre-COVID)   38.991465  -76.947101   
29  110010072001  110010072002  2018-2019 (Pre-COVID)   38.879178  -77.004542   
31  240338072001  240338072003  2018-2019 (Pre-COVID)   38.981706  -76.932857   
32  110010108002  110010108001  2018-2019 (Pre-COVID)   38.896391  -77.042766   

     dest_lat   dest_lng  avg_visitor_count  avg_median_dwell  
10  39.195798 -77.253170             1662.6              26.9  
15  38.983788 -76.941811             1401.5              41.7  
29  38.874125 -76.999970              970.1              53.2  
31  38.983788 -76.941811              957.8              40.2  
32  38.900567 -77.047470              954.6              59.9  


## 3. Divide into Pre-COVID and COVID

In [37]:
pre_covid = flows[flows['period'] == '2018-2019 (Pre-COVID)'].copy()

covid = flows[flows['period'] == '2020-2021 (COVID)'].copy()

In [38]:
print('Pre-COVID vs COVID — Summary')
print('-' * 50)
for label, df in [('Pre-COVID (2018-2019)', pre_covid), ('COVID (2020-2021)', covid)]:
    print(f'\n  {label}')
    print(f'    OD pairs            : {len(df):,}')
    print(f'    Unique origins      : {df["home_cbg"].nunique():,}')
    print(f'    Unique destinations : {df["poi_cbg"].nunique():,}')
    print(f'    Total visitors      : {df["avg_visitor_count"].sum():,.0f}')
    print(f'    Mean dwell (min)    : {df["avg_median_dwell"].mean():.1f}')

pre_total  = pre_covid['avg_visitor_count'].sum()
cov_total  = covid['avg_visitor_count'].sum()
pct_change = (cov_total - pre_total) / pre_total * 100
print(f'\n  Mobility change COVID vs Pre-COVID: {pct_change:+.1f}%')

Pre-COVID vs COVID — Summary
--------------------------------------------------

  Pre-COVID (2018-2019)
    OD pairs            : 2,171,046
    Unique origins      : 2,548
    Unique destinations : 2,488
    Total visitors      : 13,070,184
    Mean dwell (min)    : 46.8

  COVID (2020-2021)
    OD pairs            : 1,508,760
    Unique origins      : 2,548
    Unique destinations : 2,499
    Total visitors      : 8,754,131
    Mean dwell (min)    : 52.2

  Mobility change COVID vs Pre-COVID: -33.0%


## 4. Check For Threshold and Filter

In [39]:
for threshold in [5, 10, 20, 50, 100]:
    n_pre = (pre_covid['avg_visitor_count'] >= threshold).sum()
    n_cov = (covid['avg_visitor_count'] >= threshold).sum()
    pct_visitors_pre = pre_covid[pre_covid['avg_visitor_count'] >= threshold]['avg_visitor_count'].sum() / pre_covid['avg_visitor_count'].sum() * 100
    print(f'  >= {threshold:4d} visitors  →  pre={n_pre:,}  covid={n_cov:,}  captures {pct_visitors_pre:.1f}% of total volume')

  >=    5 visitors  →  pre=749,427  covid=557,095  captures 55.7% of total volume
  >=   10 visitors  →  pre=121,980  covid=76,789  captures 25.2% of total volume
  >=   20 visitors  →  pre=44,202  covid=21,903  captures 17.3% of total volume
  >=   50 visitors  →  pre=13,050  covid=6,080  captures 10.1% of total volume
  >=  100 visitors  →  pre=4,145  covid=1,825  captures 5.5% of total volume


In [40]:
pre_covid = pre_covid[pre_covid['avg_visitor_count'] > 20].copy()
covid     = covid[covid['avg_visitor_count'] > 20].copy()

print(f'Pre-COVID flows : {len(pre_covid):,}')
print(f'COVID flows     : {len(covid):,}')

Pre-COVID flows : 43,848
COVID flows     : 21,567


## 4. Export for Kepler.gl

In [ ]:
cols = ['home_cbg', 'poi_cbg', 'origin_lat', 'origin_lng',
        'dest_lat', 'dest_lng', 'avg_visitor_count', 'avg_median_dwell']

pre_covid[cols].to_csv('../data/processed/kepler_pre_covid.csv', index=False)
covid[cols].to_csv('../data/processed/kepler_covid.csv', index=False)
